In [39]:
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import seaborn as sns
import numpy as np
import pandas as pd

---
## <u>Load Dataset</u>

In [18]:
data = sns.load_dataset("diamonds")
data.info() # no missing values
data["color"].nunique() # 7 unique values
data["cut"].nunique() # 5 unique values
data["clarity"].nunique() # 8 unique values
data.head() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


---
## <u>Train Test Split</u>

In [21]:
X = data.drop(columns = ["price"])
y = data["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

,carat,cut,color,clarity,depth,table,x,y,z
26546,2.01,Good,F,SI2,58.1,64.0,8.23,8.19,4.77
9159,1.01,Very Good,E,SI2,60.0,60.0,6.57,6.49,3.92
14131,1.10,Premium,H,VS2,62.5,58.0,6.59,6.54,4.10
15757,1.50,Good,E,SI2,61.5,65.0,7.21,7.17,4.42
24632,1.52,Very Good,G,VS1,62.1,57.0,7.27,7.32,4.53


---
## <u>Feature Encoding</u>

In [32]:
# 1. get numerical and categorical columns
num_data = X_train.select_dtypes(include = ["number"]).columns
cat_data = X_train.select_dtypes(include = ["category"]).columns

# 2. make preprocessor
preprocessor = ColumnTransformer(
    transformers = [
        ("ohe", OneHotEncoder(sparse_output = False, handle_unknown = "ignore"), cat_data),
        ("scaler", StandardScaler(), num_data)
    ]
)

# 3. set output as a dataframe
preprocessor.set_output(transform="pandas")

# 4. use the preprocessor
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test) # only transform here to handle data leakage

9


---
## <u>Create, Train and predict</u>

In [34]:
xgbr_model = xgb.XGBRegressor(random_state = 42)
xgbr_model.fit(X_train, y_train)

y_train_pred = xgbr_model.predict(X_train)
y_test_pred = xgbr_model.predict(X_test)

---
## <u>Evaluate</u>

In [35]:
print("For XGBOOST Regressor (baseline) : \n")

print("For Training set : ")
print("R2_score : ", r2_score(y_train, y_train_pred))

print("\nFor Testing set : ")
print("R2_score : ", r2_score(y_test, y_test_pred))

# no need for hyper parameter tuning as the r2_score is already very high and training_score - test_score = 0.9 which is pretty good

For XGBOOST Regressor (baseline) : 

For Training set : 
R2_score :  0.9903280735015869

For Testing set : 
R2_score :  0.9807291030883789
